## 1. Imports & Load Raw Data

In [76]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os.path as path

In [77]:
df= pd.read_csv('dataset/bank-full.csv',sep=';')
y_target = 'y'
df.head(5)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


## 2. Handle Missing / Invalid Values

In [79]:
missing_feature = df.columns[df.isnull().any()].to_list()
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')
print(f'Missing Feature:\n{missing_feature}')

Missing values: 0
Duplicates: 0
Missing Feature:
[]


In [80]:
for col in missing_feature:
    missing_percentage = (df[col].isnull().sum() / len(df)) * 100
    if missing_percentage > 5.0 :
        df = df.drop(columns=col)
    else:
        if df[col].dtype == 'object':
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            skewness = df[col].skew()
            if abs(skewness) < 0.5:
                df[col] = df[col].fillna(df[col].mean()).round(2)
            else:
                df[col] = df[col].fillna(df[col].median()).round(2)

print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Missing Feature:\n{missing_feature}')

Missing values: 0
Missing Feature:
[]


## 3. Handling Duplicated

In [82]:
df = df.drop_duplicates()

jumlah_duplikat = df.duplicated().sum()
print(f"Jumlah data duplikat: {jumlah_duplikat}")

Jumlah data duplikat: 0


## 4. Analisis & Handling Outliers

In [84]:
feature_numerik = df.select_dtypes(include=np.number).columns

Q1 = df[feature_numerik].quantile(0.25)
Q3 = df[feature_numerik].quantile(0.75)
IQR = Q3-Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df.loc[((df[feature_numerik] < lower_bound) | (df[feature_numerik] > upper_bound)).any(axis=1)]
print(f"Jumlah Outlier Sebelum Dihapus: {outliers.shape[0]}")

#delete outliers
df = df.loc[((df[feature_numerik] >= lower_bound) & (df[feature_numerik] <= upper_bound)).all(axis=1)]
outliers = df.loc[((df[feature_numerik] < lower_bound) | (df[feature_numerik] > upper_bound)).any(axis=1)]
print(f"Jumlah Outlier Sesudah Dihapus: {outliers.shape[0]}")

Jumlah Outlier Sebelum Dihapus: 17018
Jumlah Outlier Sesudah Dihapus: 0


## 5. Feature Engineering

In [86]:
batas_umur = [0, 12, 25, 45, 100]
nama_label = ['Anak-anak', 'Remaja', 'Dewasa', 'Lansia']
df['umur_category'] = pd.cut(df['age'],bins=batas_umur,labels=nama_label,include_lowest=True)

bin_balance = [-np.inf, 0, 500, 2000, np.inf]
label_balance = ['Negatif', 'Rendah', 'Sedang', 'Tinggi']
df['balance_category'] = pd.cut(df['balance'], bins=bin_balance, labels=label_balance)

bin_duration = [0, 60, 180, 300, 600, np.inf]
label_duration = ['< 1 mnt', '1-3 mnt', '3-5 mnt', '5-10 mnt', '> 10 mnt']
df['duration_category'] = pd.cut(df['duration'], bins=bin_duration, labels=label_duration, include_lowest=True)

bin_pdays = [-2, 0, 7, 30, np.inf]
label_pdays = ['Belum Pernah', 'Baru (1-7 hari)', 'Sedang (8-30 hari)', 'Lama (>30 hari)']
df['pdays_category'] = pd.cut(df['pdays'], bins=bin_pdays, labels=label_pdays)

df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,...,duration,campaign,pdays,previous,poutcome,y,umur_category,balance_category,duration_category,pdays_category
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,...,261,1,-1,0,unknown,no,Lansia,Tinggi,3-5 mnt,Belum Pernah
1,44,technician,single,secondary,no,29,yes,no,unknown,5,...,151,1,-1,0,unknown,no,Dewasa,Rendah,1-3 mnt,Belum Pernah
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,...,76,1,-1,0,unknown,no,Dewasa,Rendah,1-3 mnt,Belum Pernah
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,...,92,1,-1,0,unknown,no,Lansia,Sedang,1-3 mnt,Belum Pernah
4,33,unknown,single,unknown,no,1,no,no,unknown,5,...,198,1,-1,0,unknown,no,Dewasa,Rendah,3-5 mnt,Belum Pernah


## 6.Save Cleaned Dataset

In [88]:
file_path = 'dataset/bank-full_CLEANING.csv'

if not path.exists(file_path):
    df.to_csv(file_path, index=False)
    print('File belum ada. Berhasil menyimpan dataset baru!')
else:
    print('File sudah ada. Proses penyimpanan CSV dilewati (skip)')

df.head()

File belum ada. Berhasil menyimpan dataset baru!


,age,job,marital,education,default,balance,housing,loan,contact,day,...,duration,campaign,pdays,previous,poutcome,y,umur_category,balance_category,duration_category,pdays_category
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,...,261,1,-1,0,unknown,no,Lansia,Tinggi,3-5 mnt,Belum Pernah
1,44,technician,single,secondary,no,29,yes,no,unknown,5,...,151,1,-1,0,unknown,no,Dewasa,Rendah,1-3 mnt,Belum Pernah
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,...,76,1,-1,0,unknown,no,Dewasa,Rendah,1-3 mnt,Belum Pernah
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,...,92,1,-1,0,unknown,no,Lansia,Sedang,1-3 mnt,Belum Pernah
4,33,unknown,single,unknown,no,1,no,no,unknown,5,...,198,1,-1,0,unknown,no,Dewasa,Rendah,3-5 mnt,Belum Pernah
